In [ ]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
import pandas as pd
from keras import backend as K
from keras.models import Model
from keras.regularizers import l2
from keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, concatenate, BatchNormalization, Flatten
)
from keras.callbacks import EarlyStopping
from keras.metrics import AUC, Precision, Recall
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE

In [ ]:
print(tf.__version__)

### Data preprocessing

In [ ]:
file_path = 'NRD_2016.dta'  # Replace with your .dta file path

In [ ]:
# Load only the first 100,000 rows
chunk_iterator = pd.read_stata(file_path, chunksize=2000000)

In [ ]:
import psutil

# Function to print memory usage
def print_memory_usage():
    process = psutil.Process()
    print(f"Memory usage: {process.memory_info().rss / 1e6} MB")

In [ ]:
from tqdm import tqdm

data = []

for chunk in tqdm(chunk_iterator):
    # Example: Filter rows (e.g., based on a column value)
    data.append(chunk)
    print_memory_usage()

    # Optionally save intermediate results to disk
    # filtered_chunk.to_csv("processed_chunk.csv", mode='a', header=False, index=False)

In [ ]:
# Combine all processed chunks into a single DataFrame
data = pd.concat(data, ignore_index=True)

In [ ]:
column_names = data.columns.tolist()
print(column_names)
print(data.head)

In [ ]:
outcome_var = 'MOR30'  # Define the outcome variable here, e.g., 'DIED', 'MOR30', 'REA30'

# Step 1: onvert all column names to uppercase
data.columns = data.columns.str.upper()

# Step 2: Data Preprocessing
# Filter out observations where DIED == 1 if outcome_var is REA30
if outcome_var == 'REA30' and 'DIED' in data.columns:
    data = data[data['DIED'] != 1]


# Handle missing values in the target variable
data = data.dropna(subset=[outcome_var])


# Define ICD columns
icd_columns = [f'I10_DX{i}' for i in range(1, 36)]

# Convert all ICD codes to uppercase and combine to fit the encoder
all_icd_codes = pd.Series(data[icd_columns].values.ravel()).str.upper()
encoder = LabelEncoder()
encoder.fit(all_icd_codes)

# Transform ICD codes
for col in icd_columns:
    data[col] = encoder.transform(data[col].astype(str).str.upper())

# Get the actual number of unique ICD codes
num_unique_icd_codes = len(encoder.classes_)
print("Number of unique ICD codes:", num_unique_icd_codes)

In [ ]:
# Normalize age
age_scaler = MinMaxScaler()
data['AGE'] = age_scaler.fit_transform(data[['AGE']])

# One-hot encode 'PAY1' and 'ZIPINC_QRTL' only (excluding 'RACE')
data = pd.get_dummies(data, columns=['PAY1', 'ZIPINC_QRTL'], prefix=['PAY1', 'ZIPINC_QRTL'])

# Separate features and target variable
X = data[['AGE', 'FEMALE'] + list(data.filter(regex='PAY1_').columns) + list(data.filter(regex='ZIPINC_QRTL_').columns) + icd_columns]
y = data[outcome_var]

# Check for missing values in features
print("Missing values in features:")
print(X.isnull().sum())

# Handle missing values in features (if any)
X = X.dropna()
y = y[X.index]

In [ ]:
print(X)
print(y)

In [ ]:
# Step 3: Define and Train the Neural Network

# Split the data into training and testing sets (stratify to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# Apply SMOTE to the training data only
print("Original class distribution:", pd.Series(y_train).value_counts())


# smote = SMOTE(random_state=42)
# X_train, y_train = smote.fit_resample(X_train, y_train)
# print("Resampled class distribution:", pd.Series(y_train).value_counts())


In [ ]:
# Separate the majority and minority classes
X_majority = X_train[y_train == 0]
X_minority = X_train[y_train == 1]
y_majority = y_train[y_train == 0]
y_minority = y_train[y_train == 1]
# print(X_majority)

In [ ]:
from sklearn.utils import resample
from sklearn.utils import shuffle
import numpy as np

# Separate the majority and minority classes
X_majority = X_train[y_train == 0]
X_minority = X_train[y_train == 1]
y_majority = y_train[y_train == 0]
y_minority = y_train[y_train == 1]

# Downsample the majority class
X_majority_downsampled, y_majority_downsampled = resample(
    X_majority, y_majority, 
    replace=False,                  # No resampling; we want unique samples
    n_samples=len(y_minority),      # Match the minority class count
    random_state=42
)

# Combine the downsampled data
# Combine minority class with downsampled majority class
X_train_downsampled = pd.concat([X_minority, X_majority_downsampled])
y_train_downsampled = pd.concat([y_minority, y_majority_downsampled])

X_train_downsampled, y_train_downsampled = shuffle(X_train_downsampled, y_train_downsampled, random_state=42)

In [ ]:
print("New class distribution:", pd.Series(y_train_downsampled).value_counts())

In [ ]:
print(X_train_downsampled)
print(y_train_downsampled)

In [ ]:
# Save to a new file for further use
X_train_downsampled.to_csv("data/mort/X_train_downsampled.csv", index=False)
y_train_downsampled.to_csv("data/mort/y_train_downsampled.csv", index=False)
X_test.to_csv("data/mort/X_test.csv", index=False)
y_test.to_csv("data/mort/y_test.csv", index=False)


In [ ]:
import pickle
# Save the LabelEncoder and MinMaxScaler using pickle

# Save LabelEncoder
with open('Model/mort_2016_label_encoder.pkl', 'wb') as file:
    pickle.dump(encoder, file)

# Save MinMaxScaler for 'AGE'
with open('Model/mort_2016_age_scaler.pkl', 'wb') as file:
    pickle.dump(age_scaler, file)

In [ ]:
import pandas as pd

# Load train and test datasets
X_train_downsampled = pd.read_csv('/users/xwang259/scratch/NRD/processed_data/mort/X_train_downsampled.csv')
y_train_downsampled = pd.read_csv('/users/xwang259/scratch/NRD/processed_data/mort/y_train_downsampled.csv')
X_test = pd.read_csv('/users/xwang259/scratch/NRD/processed_data/mort/X_test.csv')
y_test = pd.read_csv('/users/xwang259/scratch/NRD/processed_data/mort/y_test.csv')


In [ ]:
print(X_train_downsampled)

In [ ]:
column_names = X_train_downsampled.columns.tolist()
print(column_names)

In [ ]:
print(y_train_downsampled)

In [ ]:
import pickle
# Load the LabelEncoder for ICD codes
with open('./Model/full_label_encoder.pkl', 'rb') as file:
    encoder = pickle.load(file)

In [ ]:
# Get the actual number of unique ICD codes
num_unique_icd_codes = len(encoder.classes_)
print("Number of unique ICD codes:", num_unique_icd_codes)

In [ ]:
from keras.saving import register_keras_serializable


### Transformers

In [ ]:
import tensorflow as tf
from keras.layers import Input, Dense, Dropout, BatchNormalization, Embedding, Flatten, concatenate, MultiHeadAttention, LayerNormalization, Add
from keras.models import Model
from keras.regularizers import l2
from keras.callbacks import EarlyStopping
from keras.metrics import AUC, Precision, Recall
import keras.backend as K

# Custom Transformer Encoder Block
@tf.keras.utils.register_keras_serializable(package="Custom")
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

    def get_config(self):
        # Return the parameters required for serialization
        config = super(TransformerBlock, self).get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "rate": self.rate
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Recreate the layer from its config
        return cls(**config)
    

# class ResidualBlock(tf.keras.layers.Layer):
#     def __init__(self, units, dropout_rate=0.2):
#         super(ResidualBlock, self).__init__()
#         self.dense1 = Dense(units, activation='relu', kernel_regularizer=l2(0.001))
#         self.batch_norm1 = BatchNormalization()
#         self.dropout1 = Dropout(dropout_rate)

#         self.dense2 = Dense(units, kernel_regularizer=l2(0.001))
#         self.batch_norm2 = BatchNormalization()
#         self.dropout2 = Dropout(dropout_rate)

#         self.shortcut_dense = Dense(units) if units else None  # Use a dense layer if needed to match dimensions

#     def call(self, inputs, training=False):
#         x = self.dense1(inputs)
#         x = self.batch_norm1(x, training=training)
#         x = self.dropout1(x, training=training)

#         x = self.dense2(x)
#         x = self.batch_norm2(x, training=training)
#         x = self.dropout2(x, training=training)

#         # Adjust input dimensions if necessary
#         shortcut = self.shortcut_dense(inputs) if self.shortcut_dense else inputs
#         return tf.keras.layers.Add()([x, shortcut])   

In [ ]:
# @tf.keras.utils.register_keras_serializable(package="Custom")
# class DeepSet(tf.keras.Model):
#     def __init__(self, input_dim, hidden_dim, output_dim, **kwargs):
#         super(DeepSet, self).__init__(**kwargs)
        
#         self.input_dim = input_dim
#         self.hidden_dim = hidden_dim
#         self.output_dim = output_dim
#         # Element-wise transformation: phi network
#         self.phi = tf.keras.Sequential([
#             Dense(self.hidden_dim, activation='relu')
#         ])
#         # Post-aggregation transformation: rho network
#         self.rho = tf.keras.Sequential([
#             Dense(self.output_dim, activation='relu')
#         ])

    
#     def call(self, x):
#         # Apply phi to each ICD code embedding
#         transformed = self.phi(x)  # Shape: (batch_size, num_codes, output_dim)
#         # Aggregate using sum (or other aggregation functions)
#         aggregated = tf.reduce_sum(transformed, axis=1)  # Shape: (batch_size, output_dim)
#         # Apply rho to the aggregated representation
#         output = self.rho(aggregated)  # Shape: (batch_size, output_dim)
#         return output

#     def get_config(self):
#         # Return the parameters required for serialization
#         config = super(DeepSet, self).get_config()
#         config.update({
#             "input_dim": self.input_dim,
#             "hidden_dim": self.hidden_dim,
#             "output_dim": self.output_dim
#         })
#         return config

#     @classmethod
#     def from_config(cls, config):
#         # Recreate the layer from its config
#         return cls(**config)


In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom")
class DeepSet(tf.keras.Model):
    def __init__(self, input_dim, hidden_dim, output_dim, num_encode, num_decode, **kwargs):
        super(DeepSet, self).__init__(**kwargs)
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.num_encode = num_encode
        self.num_decode = num_decode
        # # Element-wise transformation: phi network
        # self.phi = tf.keras.Sequential([
        #     Dense(self.hidden_dim, activation='relu')
        # ])
        # # Post-aggregation transformation: rho network
        # self.rho = tf.keras.Sequential([
        #     Dense(self.output_dim, activation='relu')
        # ])

        # Element-wise transformation: phi network
        self.phi = tf.keras.Sequential([
            Dense(self.hidden_dim, activation='relu') for _ in range(self.num_encode)
        ])

        # Post-aggregation transformation: rho network
        self.rho = tf.keras.Sequential([
            Dense(self.hidden_dim, activation='relu') for _ in range(self.num_decode - 1)
        ] + [Dense(self.output_dim, activation='relu')])  # Last layer should output correct dimension

    def call(self, x):
        transformed = self.phi(x)  # (batch_size, num_codes, hidden_dim)
        aggregated = tf.reduce_sum(transformed, axis=1)  # (batch_size, hidden_dim)
        output = self.rho(aggregated)  # (batch_size, output_dim)
        return output

    def get_config(self):
        config = super(DeepSet, self).get_config()
        config.update({
            "input_dim": self.input_dim,
            "hidden_dim": self.hidden_dim,
            "output_dim": self.output_dim,
            "num_encode": self.num_encode,
            "num_decode": self.num_decode
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [ ]:

@tf.keras.utils.register_keras_serializable(package="Custom")
class F2Score(tf.keras.metrics.Metric):
    def __init__(self, name='f2_score', threshold=0.5, **kwargs):
        super(F2Score, self).__init__(name=name, **kwargs)
        self.tp = self.add_weight(name='tp', initializer='zeros')
        self.fp = self.add_weight(name='fp', initializer='zeros')
        self.fn = self.add_weight(name='fn', initializer='zeros')
        self.epsilon = tf.keras.backend.epsilon()
        self.threshold = threshold

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        self.tp.assign_add(tf.reduce_sum(y_true * y_pred))
        self.fp.assign_add(tf.reduce_sum((1 - y_true) * y_pred))
        self.fn.assign_add(tf.reduce_sum(y_true * (1 - y_pred)))

    def result(self):
        precision = self.tp / (self.tp + self.fp + self.epsilon)
        recall = self.tp / (self.tp + self.fn + self.epsilon)
        f2 = (5 * precision * recall) / (4 * precision + recall + self.epsilon)
        return f2

    def reset_state(self, sample_weight=None):
        self.tp.assign(0.0)
        self.fp.assign(0.0)
        self.fn.assign(0.0)

    def get_config(self):
        config = super(F2Score, self).get_config()
        config.update({'name': self.name, 'threshold': self.threshold})
        return config

    @classmethod
    def from_config(cls, config):
        # Recreate the layer from its config
        return cls(**config)

@tf.keras.utils.register_keras_serializable(package="Custom")
def f2_score(y_true, y_pred):
    # y_pred = tf.cast(y_pred > 0.5, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    epsilon = tf.keras.backend.epsilon()
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    f2 = (5 * precision * recall) / (4 * precision + recall + epsilon)
    return f2

In [ ]:
icd_columns = [f'I10_DX{i}' for i in range(1, 41)]

In [ ]:
# # Define the ICD code input layer with trainable embeddings
# icd_inputs = Input(shape=(len(icd_columns),), name='icd_codes')
# icd_embedding = Embedding(
#     input_dim=num_unique_icd_codes,
#     output_dim=32,
#     name='icd_embedding',
#     trainable=True
# )(icd_inputs)
# print(icd_embedding.shape)

# x = icd_embedding

# Add Transformer layer to ICD embeddings
# num_transformer_blocks = ４  # Set the number of transformer blocks to stack
# for i in range(num_transformer_blocks):
#     transformer_block = TransformerBlock(embed_dim=32, num_heads=3, ff_dim=128, rate=0.4)
#     x = transformer_block(x, training=True)

# deep_set_output_dim = int(160 * 0.9)
# agg_block = DeepSet(input_dim = 64, hidden_dim = 160, output_dim = deep_set_output_dim, num_encode = 2, num_decode = 3)
# x = agg_block(x) 
    
# # Flatten the transformer output for input into the MLP
# x = Flatten()(icd_embedding)

# # Define MLP layers after transformer encoder
# # MLP layers for ICD embeddings
# x = Dense(256, activation='relu', kernel_regularizer=l2(0.001), name='mlp_dense_1')(x)
# x = Dropout(0.4, name='mlp_dropout_1')(x)
# x = Dense(128, activation='relu', kernel_regularizer=l2(0.001), name='mlp_dense_2')(x)
# x = Dropout(0.4, name='mlp_dropout_2')(x)
# x = Dense(64, activation='relu', kernel_regularizer=l2(0.001), name='mlp_dense_3')(x)

# Define demographic and one-hot encoded inputs
age_input = Input(shape=(1,), name='age')
female_input = Input(shape=(1,), name='female')
pay1_inputs = [Input(shape=(1,), name=f'PAY1_{col}') for col in X_train_downsampled.filter(regex='PAY1_').columns]
zipinc_qrtl_inputs = [Input(shape=(1,), name=f'ZIPINC_QRTL_{col}') for col in X_train_downsampled.filter(regex='ZIPINC_QRTL_').columns]

# Concatenate demographic inputs
demographic_inputs = [age_input, female_input] + pay1_inputs + zipinc_qrtl_inputs
demographic_concat = concatenate(demographic_inputs, name='demographic_concat')

# Process demographics through a small MLP
demographic_hidden = Dense(64, activation='relu', name='demographic_dense1')(demographic_concat)
demographic_hidden = Dense(32, activation='relu', name='demographic_dense2')(demographic_hidden)

# Concatenate all inputs
# concatenated = concatenate([x, age_input, female_input] + pay1_inputs + zipinc_qrtl_inputs, name='concatenate')
# concatenated = x
concatenated = demographic_hidden

# Add BatchNormalization and dense layers
hidden = BatchNormalization(name='batch_norm')(concatenated)

for i in range(4):        
    units = max(32, int(512 * (0.5 ** i)))
    hidden = Dense(units, activation='relu', kernel_regularizer=l2(0.001), name=f'dense_{i}')(hidden)
    hidden = Dropout(0.4, name=f'dropout_{i}')(hidden)

# hidden = BatchNormalization(name='batch_norm')(concatenated)
# hidden = ResidualBlock(units=128, dropout_rate=0.3)(hidden)
# hidden = ResidualBlock(units=64, dropout_rate=0.3)(hidden)
# hidden = ResidualBlock(units=32, dropout_rate=0.2)(hidden)

# Output layer for mortality prediction
output = Dense(1, activation='sigmoid', name='output')(hidden)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Assuming y_train contains your labels (0 for negative, 1 for positive)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

print(class_weight_dict)

In [ ]:
@register_keras_serializable()
def weighted_binary_crossentropy(y_true, y_pred):
    # Convert the class weights into a tensor of the same shape as y_true
    weights = tf.where(
        tf.equal(y_true, 1),  # Condition: True if y_true is 1 (positive class)
        tf.constant(class_weight_dict[1], dtype=tf.float32),  # Weight for positive class
        tf.constant(class_weight_dict[0], dtype=tf.float32)   # Weight for negative class
    )
    
    # Compute binary cross-entropy
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    
    # Return the weighted binary cross-entropy
    return tf.reduce_mean(bce * weights)

In [ ]:
import keras

# Build the model
# model = Model(inputs=[icd_inputs, age_input, female_input] + pay1_inputs + zipinc_qrtl_inputs, outputs=output)
model = Model(inputs=[age_input, female_input] + pay1_inputs + zipinc_qrtl_inputs, outputs=output)
# model = Model(inputs=[icd_inputs], outputs=output)

# Compile the model with the weighted loss function
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.binary_crossentropy,  # Use the custom weighted loss
    metrics=[AUC(name='auc'), Precision(name='precision'), Recall(name='recall'), F2Score]
)

# Implement early stopping
early_stopping = EarlyStopping(
    monitor='val_f2_score', patience=3, mode='max', restore_best_weights=True
)

# # Train the model
# history = model.fit(
#     [X_train_downsampled[icd_columns], X_train_downsampled['AGE'], X_train_downsampled['FEMALE']] + [X_train_downsampled[col] for col in X_train_downsampled.filter(regex='PAY1_').columns] + [X_train_downsampled[col] for col in X_train_downsampled.filter(regex='ZIPINC_QRTL_').columns],
#     y_train_downsampled,
#     epochs=10,
#     batch_size=128,
#     validation_split=0.1,
#     callbacks=[early_stopping]
# )

# Train the model
history = model.fit(
    [X_train_downsampled['AGE'], X_train_downsampled['FEMALE']] + [X_train_downsampled[col] for col in X_train_downsampled.filter(regex='PAY1_').columns] + [X_train_downsampled[col] for col in X_train_downsampled.filter(regex='ZIPINC_QRTL_').columns],
    y_train_downsampled,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stopping]
)

# # Train the model
# history = model.fit(
#     [X_train_downsampled[icd_columns]],
#     y_train_downsampled,
#     epochs=10,
#     batch_size=128,
#     validation_split=0.1,
#     callbacks=[early_stopping]
# )

In [ ]:
print(X_test)

In [ ]:

# # Step 4: Evaluate the model
# test_loss, test_auc, test_precision, test_recall, test_f2 = model.evaluate(
#     [X_test[icd_columns], X_test['AGE'], X_test['FEMALE']] + [X_test[col] for col in X_test.filter(regex='PAY1_').columns] + [X_test[col] for col in X_test.filter(regex='ZIPINC_QRTL_').columns], 
#     y_test)

# Step 4: Evaluate the model
test_loss, test_auc, test_precision, test_recall, test_f2 = model.evaluate(
    [X_test['AGE'], X_test['FEMALE']] + [X_test[col] for col in X_test.filter(regex='PAY1_').columns] + [X_test[col] for col in X_test.filter(regex='ZIPINC_QRTL_').columns], 
    y_test)

print(f'Test AUC: {test_auc:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall: {test_recall:.4f}')
print(f'Test F2 Score: {test_f2:.4f}')


In [ ]:
# Calculate F1-score
y_pred_prob = model.predict(
    [X_test[icd_columns], X_test['AGE'], X_test['FEMALE']] + [X_test[col] for col in data.filter(regex='PAY1_').columns] + [X_test[col] for col in data.filter(regex='ZIPINC_QRTL_').columns]
)
print(y_pred_prob)


In [ ]:
optimal_threshold = 0
best_f1 = 0
for threshold in np.linspace(0, 1, 100):
    preds = (y_pred_prob > threshold).astype(int)
    score = f1_score(y_test, preds)
    if score > best_f1:
        best_f1 = score
        optimal_threshold = threshold
print(optimal_threshold)
print(best_f1)

In [ ]:

# Calculate F1-score
y_pred_prob = model.predict(
    [X_test[icd_columns], X_test['AGE'], X_test['FEMALE']] + [X_test[col] for col in data.filter(regex='PAY1_').columns] + [X_test[col] for col in data.filter(regex='ZIPINC_QRTL_').columns]
)
y_pred = (y_pred_prob > 0.5).astype(int)

f1 = f1_score(y_test, y_pred)
print(f'Test F1 Score: {f1:.4f}')

In [ ]:
import pickle
import keras

# # Save the trained model
model.save('Model/readmit_full_trial.keras')

# keras.saving.save_model(model, 'Model/mort_2016.keras')